In [ ]:
#!pip install gudhi

In [ ]:
# Standard scientific Python imports
import numpy as np

# Standard scikit-learn imports
from sklearn.datasets import fetch_openml
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn import metrics
from sklearn.preprocessing import MinMaxScaler


# Import TDA pipeline requirements
from gudhi.sklearn.cubical_persistence import CubicalPersistence
from gudhi.representations import PersistenceImage, DiagramSelector

X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)


In [ ]:
#pick up 1000 samples
X = X[:1000]
y = y[:1000]

X_train, X_test, y_train, y_test = train_test_split(X/255, y, test_size=0.2, random_state=0)

clf1 = SVC()
clf1.fit(X_train, y_train)
predicted = clf1.predict(X_test)

In [ ]:
print(f"Classification report SVC {clf1}:\n" f"{metrics.classification_report(y_test, predicted)}\n")

report = metrics.classification_report(y_test, predicted, output_dict=True)

Classification report SVC SVC():
              precision    recall  f1-score   support

           0       0.89      1.00      0.94        17
           1       0.93      1.00      0.96        25
           2       0.84      0.84      0.84        19
           3       1.00      0.93      0.97        15
           4       0.95      1.00      0.98        20
           5       0.76      0.90      0.83        21
           6       0.94      0.89      0.91        18
           7       0.96      0.92      0.94        24
           8       0.95      0.82      0.88        22
           9       0.94      0.79      0.86        19

    accuracy                           0.91       200
   macro avg       0.92      0.91      0.91       200
weighted avg       0.91      0.91      0.91       200




In [ ]:
pipe = Pipeline(
    [
        ("cub_pers", CubicalPersistence(homology_dimensions=0, n_jobs=-2)),
        # Or for multiple persistence dimension computation
        # ("cub_pers", CubicalPersistence(homology_dimensions=[0, 1])),
        # ("H0_diags", DimensionSelector(index=0), # where index is the index in homology_dimensions array
        ("finite_diags", DiagramSelector(use=True, point_type="finite")),
        (
            "pers_img",
            PersistenceImage(bandwidth=50, weight=lambda x: x[1] ** 2, im_range=[0, 256, 0, 256], resolution=[20, 20]),
        ),
        ("standard_scaler", MinMaxScaler()),
    ]
)

In [ ]:
X_trans = pipe.fit_transform(X)


In [ ]:
X_trans

array([[0.06104277, 0.0738459 , 0.09448078, ..., 0.05031928, 0.0461703 ,
        0.05123464],
       [0.09981941, 0.11046449, 0.12337841, ..., 0.62239365, 0.60363498,
        0.58910625],
       [0.16317873, 0.16270297, 0.1609747 , ..., 0.05577042, 0.03758827,
        0.02785255],
       ...,
       [0.00216549, 0.00352682, 0.00606016, ..., 0.23874844, 0.21133676,
        0.19431616],
       [0.31718323, 0.34233455, 0.37467116, ..., 0.06941222, 0.05510566,
        0.05216889],
       [0.09532934, 0.09797684, 0.1013877 , ..., 0.55989044, 0.49823512,
        0.44920071]])

In [ ]:
X_sum = np.hstack((X/255, X_trans))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_sum, y, test_size=0.2, random_state=0)



In [ ]:
clf = SVC()
clf.fit(X_train, y_train)
predicted = clf.predict(X_test)

In [ ]:
print(f"Classification report SVC_tda {clf}:\n" f"{metrics.classification_report(y_test, predicted)}\n")

report = metrics.classification_report(y_test, predicted, output_dict=True)

Classification report SVC_tda SVC():
              precision    recall  f1-score   support

           0       0.85      1.00      0.92        17
           1       0.96      1.00      0.98        25
           2       0.89      0.89      0.89        19
           3       1.00      1.00      1.00        15
           4       1.00      0.95      0.97        20
           5       0.86      0.90      0.88        21
           6       0.94      0.94      0.94        18
           7       0.96      0.96      0.96        24
           8       1.00      0.91      0.95        22
           9       0.94      0.84      0.89        19

    accuracy                           0.94       200
   macro avg       0.94      0.94      0.94       200
weighted avg       0.94      0.94      0.94       200


